# bce-log-loss-real-fake — worked example 2: Non-saturating generator loss -log(D(fake))

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `bce-log-loss-real-fake`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

The generator's half of the minimax game flips the fake target. Instead of minimizing `log(1 - D(fake))` (which saturates and gives weak gradients early in training), the practical DCGAN generator MAXIMIZES `log(D(fake))`, i.e. minimizes `-log(D(fake))`. That is exactly `BCE(D(fake), 1)` — the generator wants the discriminator to call its fakes real.

## Worked solution

**Goal:** compute the non-saturating generator loss `-log(D(fake)).mean()`.

1. **The generator's wish.** G succeeds when D is fooled, i.e. when `D(fake)` is close to 1. So G's per-sample penalty is `-log(d_pred_fake)`, small when D is fooled, large when D is not.
2. **Why not `log(1 - D(fake))`.** The original minimax form has G minimize `log(1 - D(fake))`. Early in training D easily rejects fakes (`D(fake) ≈ 0`), where that term is flat — vanishing gradient. The `-log(D(fake))` form has steep gradient exactly there, so it is the one used in practice.
3. **Target of all ones.** Because G wants `D(fake) = 1`, the BCE target is `ones_like(d_pred_fake)`, NOT zeros. This is the single sign-flip that distinguishes the generator step from the discriminator's fake term.
4. **Mean over the batch** gives a scalar.
5. **Cross-check.** `F.binary_cross_entropy(d_pred_fake, ones_like)` equals `-log(d_pred_fake).mean()` by definition of BCE at target 1.

In [ ]:
import torch.nn.functional as F

def generator_loss(d_pred_fake: t.Tensor) -> t.Tensor:
    eps = 1e-7
    pf = d_pred_fake.clamp(eps, 1 - eps)
    return -t.log(pf).mean()

t.manual_seed(0)
d_pred_fake = t.rand(8)

manual = generator_loss(d_pred_fake)
bce = F.binary_cross_entropy(d_pred_fake, t.ones_like(d_pred_fake))
print('generator -log(D(fake)) loss:', round(manual.item(), 6))
print('BCE(D(fake), 1):', round(bce.item(), 6))
print('match:', t.allclose(manual, bce, atol=1e-5))